# Meridian - training pipeline

One notebook: build the routing dataset **and** train the router. Runs on a
single Colab / Kaggle GPU.

```
public datasets  ->  ~500 prompts
      ->  answers from Qwen3-0.6B / 1.7B / 8B   (vLLM, one subprocess per model)
      ->  Qwen2.5-14B-Instruct judge scores each answer 0-10
      ->  "cheapest good-enough" routing label
      ->  push  srmty/routing_dataset
      ->  fine-tune microsoft/deberta-v3-small (3-class)
      ->  evaluate (accuracy / P / R / F1 / confusion matrix / confidence)
      ->  push  srmty/meridian
```

Generation and judging run with **vLLM**, and each model runs in its **own
`python` subprocess** (`gen_worker.py` / `judge_worker.py`) - the clean way to
cycle several models through one Colab GPU, since the OS reclaims all the VRAM
when each process exits. Qwen3-8B and the 14B judge use `-AWQ` 4-bit checkpoints,
so it fits ~15 GB (a free T4) and 500 prompts x 3 models takes ~30-45 min. The
judge can also run remotely (`JUDGE_LOCAL = False` -> featherless-ai). With
`JUDGE_LOCAL = True` the notebook makes **no inference-API calls at all** - only
dataset downloads and the two `push_to_hub` calls need the network.

The **demo app never downloads a Qwen model** - it always routes to them through
the HF inference router. Only this training notebook runs them locally.

Every stage checkpoints to `checkpoints/` and resumes.

## 0. Setup

In [ ]:
%pip install -q vllm

_cu = !python -c "import torch; print('cu' + torch.version.cuda.replace('.', ''))"
_cu = _cu[-1].strip()
print("matching torch CUDA build:", _cu)
!pip install -q --force-reinstall --no-deps torchvision torchaudio --index-url https://download.pytorch.org/whl/{_cu}

%pip install -q datasets accelerate sentencepiece scikit-learn huggingface_hub openai


### 0b. Sanity check

Run this right after restarting. It imports torch / torchvision / torchaudio /
vLLM **in a subprocess** - exactly how the worker scripts do - so a CUDA mismatch
or missing package shows up here in 5 seconds instead of 20 minutes into
generation. All three of torch / torchvision / torchaudio must report the same
CUDA (13.0).

In [ ]:
!python -c "import torch, torchvision, torchaudio, vllm; print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| torchaudio', torchaudio.__version__, '| vllm', vllm.__version__); print('cuda:', torch.version.cuda, '| gpu:', torch.cuda.get_device_name(0))"


In [ ]:
import gc, json, os, re, subprocess, sys, time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42

QUALITY_THRESHOLD = 8.0
ROUTER_THRESHOLD  = 0.75

QWEN_HF = {
    "small":  "Qwen/Qwen3-0.6B",
    "medium": "Qwen/Qwen3-1.7B",
    "large":  "Qwen/Qwen3-8B-AWQ",
}
GEN_MAX_NEW_TOKENS = 256

JUDGE_LOCAL  = True
JUDGE_HF     = "Qwen/Qwen2.5-14B-Instruct-AWQ"
JUDGE_REMOTE = "Qwen/Qwen2.5-14B-Instruct:featherless-ai"

DATASET_REPO = "srmty/routing_dataset"
ROUTER_REPO  = "srmty/meridian"
BASE_MODEL   = "microsoft/deberta-v3-small"

N_PER_SOURCE = {"gsm8k": 85, "humaneval": 85, "mmlu": 85, "mbpp": 85, "hotpotqa": 80, "cnndm": 80}

LIMIT = None

CKPT_DIR = Path("checkpoints")
CKPT_DIR.mkdir(exist_ok=True)

LABELS = ["small", "medium", "large"]
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}


In [ ]:
from huggingface_hub import login

if "HF_TOKEN" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass

login(token=os.environ.get("HF_TOKEN"))
assert os.environ.get("HF_TOKEN"), "set HF_TOKEN (Colab secret or env var) before continuing"


## 1. Load public datasets -> ~500 prompts

Balanced mix across math, code, knowledge, multi-hop QA and summarization. We
keep only `id`, `prompt`, `category` - the answers come later.

In [ ]:
from datasets import load_dataset

def _load(name, config, split):
    kw = {"split": split}
    if config:
        kw["name"] = config
    try:
        return load_dataset(name, **kw)
    except Exception as e:
        print(f"  {name}: retrying with trust_remote_code=True ({e})")
        return load_dataset(name, trust_remote_code=True, **kw)

def sample(name, config, split, n, fmt, category):
    ds = _load(name, config, split)
    ds = ds.shuffle(seed=SEED).select(range(min(n, len(ds))))
    return [{"prompt": fmt(r), "category": category} for r in ds]

def f_mmlu(r):
    opts = "\n".join(f"{chr(65+i)}. {c}" for i, c in enumerate(r["choices"]))
    return f"{r['question']}\n{opts}\nAnswer with the correct option letter and a one-line justification."

pool  = sample("openai/gsm8k", "main", "train", N_PER_SOURCE["gsm8k"],
               lambda r: r["question"], "math")
pool += sample("openai/openai_humaneval", None, "test", N_PER_SOURCE["humaneval"],
               lambda r: "Complete this Python function:\n\n" + r["prompt"], "code")
pool += sample("cais/mmlu", "all", "test", N_PER_SOURCE["mmlu"], f_mmlu, "knowledge")
pool += sample("google-research-datasets/mbpp", "full", "train", N_PER_SOURCE["mbpp"],
               lambda r: r["text"] + "\n\nWrite a single Python function that solves this.", "code")
pool += sample("hotpotqa/hotpot_qa", "distractor", "validation", N_PER_SOURCE["hotpotqa"],
               lambda r: "Answer concisely: " + r["question"], "qa")
pool += sample("abisee/cnn_dailymail", "3.0.0", "train", N_PER_SOURCE["cnndm"],
               lambda r: "Summarize this article in 3-4 sentences:\n\n" + r["article"][:1500], "summarization")

import random
random.Random(SEED).shuffle(pool)

records = []
for i, r in enumerate(pool, start=1):
    records.append({"id": i, "prompt": r["prompt"].strip()[:2000], "category": r["category"]})

if LIMIT:
    records = records[:LIMIT]

(CKPT_DIR / "prompts.jsonl").write_text("\n".join(json.dumps(r) for r in records))
print(len(records), "prompts")
pd.Series([r["category"] for r in records]).value_counts()


In [ ]:
records[0]


## 2. Generate answers - Qwen3 0.6B / 1.7B / 8B with vLLM

Each model runs in its **own `python` process** (`gen_worker.py`), one after the
other. That is the reliable way to cycle three models through one Colab GPU: when
a process exits the OS reclaims 100% of the VRAM - no in-notebook `del` /
`empty_cache()` guesswork, and no vLLM-in-Jupyter stdout issues.

vLLM's continuous batching does ~500 prompts in a couple of minutes per model.
`enable_thinking=False` + a `<think>` strip keeps Qwen3's reasoning trace out of
the data. The worker appends to `checkpoints/gen_<size>.jsonl` every 128 prompts
and skips ids already there, so a disconnect only costs the current chunk.

In [ ]:
%%writefile gen_worker.py
import json, os, re, sys
from pathlib import Path

os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)

def main():
    model, prompts, out, max_tokens = sys.argv[1], Path(sys.argv[2]), Path(sys.argv[3]), int(sys.argv[4])
    print(f"[gen_worker] model={model} prompts={prompts} out={out} max_new={max_tokens}", flush=True)

    rows = [json.loads(l) for l in prompts.read_text().splitlines() if l.strip()]
    seen = {json.loads(l)["id"] for l in out.read_text().splitlines() if l.strip()} if out.exists() else set()
    rows = [r for r in rows if r["id"] not in seen]
    print(f"{model}: {len(seen)} done, {len(rows)} to generate", flush=True)
    if not rows:
        return

    from vllm import LLM, SamplingParams
    llm = LLM(model=model, dtype="float16", gpu_memory_utilization=0.85,
              max_model_len=4096, enforce_eager=True)
    sp = SamplingParams(temperature=0.7, top_p=0.8, top_k=20, max_tokens=max_tokens)

    CH = 128
    with out.open("a") as f:
        for i in range(0, len(rows), CH):
            chunk = rows[i:i + CH]
            outs = llm.chat([[{"role": "user", "content": r["prompt"]}] for r in chunk], sp,
                            chat_template_kwargs={"enable_thinking": False})
            for r, o in zip(chunk, outs):
                text = _THINK_RE.sub("", o.outputs[0].text).strip()
                f.write(json.dumps({"id": r["id"], "response": text}) + "\n")
            f.flush()
            print(f"  {min(i + CH, len(rows))}/{len(rows)}", flush=True)
    print(f"done -> {out}", flush=True)

if __name__ == "__main__":
    main()


In [ ]:
%%writefile judge_worker.py
import json, os, re, sys
from pathlib import Path

os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

JUDGE_SYS = "You are a strict evaluator. Output JSON only. No prose, no code fences."
JUDGE_TMPL = (
    "Judge the three answers to the PROMPT. Weigh correctness, relevance, "
    "completeness and clarity, then give ONE overall 0-10 score per answer.\n\n"
    "PROMPT:\n{prompt}\n\n"
    "ANSWER A - Qwen3-0.6B:\n{small}\n\n"
    "ANSWER B - Qwen3-1.7B:\n{medium}\n\n"
    "ANSWER C - Qwen3-8B:\n{large}\n\n"
    "Respond with exactly this JSON and nothing else:\n"
    '{{"small_score": <float 0-10>, "medium_score": <float 0-10>, "large_score": <float 0-10>}}'
)
_JSON_RE = re.compile(r"\{[^{}]*\}", re.DOTALL)

def _clamp(x):
    return float(max(0.0, min(10.0, float(x))))

def build_messages(row):
    user = JUDGE_TMPL.format(
        prompt=row["prompt"][:2000],
        small=row["small_response"][:2000] or "(empty)",
        medium=row["medium_response"][:2000] or "(empty)",
        large=row["large_response"][:2000] or "(empty)",
    )
    return [{"role": "system", "content": JUDGE_SYS}, {"role": "user", "content": user}]

def parse_scores(text):
    m = _JSON_RE.search(text or "")
    if not m:
        return None
    try:
        d = json.loads(m.group(0))
        return {"small_score": _clamp(d["small_score"]),
                "medium_score": _clamp(d["medium_score"]),
                "large_score": _clamp(d["large_score"])}
    except Exception:
        return None

def main():
    model, inp, out = sys.argv[1], Path(sys.argv[2]), Path(sys.argv[3])
    print(f"[judge_worker] model={model} in={inp} out={out}", flush=True)
    rows = [json.loads(l) for l in inp.read_text().splitlines() if l.strip()]
    seen = {json.loads(l)["id"] for l in out.read_text().splitlines() if l.strip()} if out.exists() else set()
    rows = [r for r in rows if r["id"] not in seen]
    print(f"{len(seen)} done, {len(rows)} to judge", flush=True)
    if not rows:
        return

    from vllm import LLM, SamplingParams
    llm = LLM(model=model, dtype="float16", gpu_memory_utilization=0.85,
              max_model_len=4096, enforce_eager=True)
    sp = SamplingParams(temperature=0.0, max_tokens=200)

    CH = 128
    with out.open("a") as f:
        for i in range(0, len(rows), CH):
            chunk = rows[i:i + CH]
            outs = llm.chat([build_messages(r) for r in chunk], sp)
            for r, o in zip(chunk, outs):
                s = parse_scores(o.outputs[0].text)
                if s:
                    f.write(json.dumps({**r, **s}) + "\n")
                else:
                    print(f"  id={r['id']} unparseable, skipped", flush=True)
            f.flush()
            print(f"  {min(i + CH, len(rows))}/{len(rows)}", flush=True)
    print(f"done -> {out}", flush=True)

if __name__ == "__main__":
    main()


In [ ]:
def run_worker(args):
    p = subprocess.Popen([sys.executable, "-u", *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if p.returncode:
        raise RuntimeError(f"{args[0]} exited {p.returncode} - see output above")


In [ ]:
assert Path("gen_worker.py").exists(), "run the `%%writefile gen_worker.py` cell first"
assert "__main__" in Path("gen_worker.py").read_text(), "gen_worker.py is stale - re-run its %%writefile cell"

(CKPT_DIR / "prompts.jsonl").write_text("\n".join(json.dumps(r) for r in records))
print(len(records), "prompts ->", CKPT_DIR / "prompts.jsonl")

for size in LABELS:
    print(f"\n=== {size}: {QWEN_HF[size]} ===", flush=True)
    run_worker(["gen_worker.py", QWEN_HF[size],
                str(CKPT_DIR / "prompts.jsonl"), str(CKPT_DIR / f"gen_{size}.jsonl"),
                str(GEN_MAX_NEW_TOKENS)])

gen = {s: {json.loads(l)["id"]: json.loads(l)["response"]
           for l in (CKPT_DIR / f"gen_{s}.jsonl").read_text().splitlines() if l.strip()}
       for s in LABELS}

answered = [{**rec,
             "small_response":  gen["small"][rec["id"]],
             "medium_response": gen["medium"][rec["id"]],
             "large_response":  gen["large"][rec["id"]]}
            for rec in records if all(rec["id"] in gen[s] for s in LABELS)]

(CKPT_DIR / "answered.jsonl").write_text("\n".join(json.dumps(r) for r in answered))
print(len(answered), "rows with all three answers")
answered[0]


## 3. Judge the answers - score each 0-10

`Qwen2.5-14B-Instruct` reads the prompt + all three answers and returns strict
JSON. `JUDGE_LOCAL` (cell 3):

- **`True` (default)** - `judge_worker.py` runs `Qwen/Qwen2.5-14B-Instruct-AWQ`
  with vLLM in its own process (fits a T4, ~10-15 min for 500, no API calls).
- **`False`** - `Qwen/Qwen2.5-14B-Instruct:featherless-ai` via the HF router
  (no local VRAM; spends HF inference credits).

```json
{"small_score": 7.0, "medium_score": 8.5, "large_score": 9.0}
```

In [ ]:
JUDGED_PATH = CKPT_DIR / "judged.jsonl"

if JUDGE_LOCAL:
    run_worker(["judge_worker.py", JUDGE_HF,
                str(CKPT_DIR / "answered.jsonl"), str(JUDGED_PATH)])
else:
    sys.path.insert(0, ".")
    from judge_worker import build_messages, parse_scores
    from openai import OpenAI

    seen = ({json.loads(l)["id"] for l in JUDGED_PATH.read_text().splitlines() if l.strip()}
            if JUDGED_PATH.exists() else set())
    todo = [r for r in answered if r["id"] not in seen]
    print(f"judge (remote): {len(seen)} done, {len(todo)} to do")
    client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=os.environ["HF_TOKEN"])
    with JUDGED_PATH.open("a") as f:
        for r in tqdm(todo, desc="judging (remote)"):
            raw = ""
            for attempt in range(4):
                try:
                    raw = client.chat.completions.create(
                        model=JUDGE_REMOTE, messages=build_messages(r),
                        max_tokens=200, temperature=0.0,
                    ).choices[0].message.content or ""
                    break
                except Exception as e:
                    print(f"  id={r['id']} attempt {attempt+1}: {e}")
                    time.sleep(2 ** attempt)
            s = parse_scores(raw)
            if s:
                f.write(json.dumps({**r, **s}) + "\n")
            else:
                print(f"  id={r['id']} judge failed, skipping")

judged = {json.loads(l)["id"]: json.loads(l)
          for l in JUDGED_PATH.read_text().splitlines() if l.strip()}
judged_rows = [judged[r["id"]] for r in answered if r["id"] in judged]
print(len(judged_rows), "rows judged")
{k: judged_rows[0][k] for k in ("id", "small_score", "medium_score", "large_score")}


## 4. Routing label = cheapest good-enough model

Not "which score is highest" - "which is the smallest model that already clears
the bar".

```
if small_score  >= QUALITY_THRESHOLD:  label = small
elif medium_score >= QUALITY_THRESHOLD: label = medium
else:                                   label = large
```

In [ ]:
def make_label(s, m, l):
    if s >= QUALITY_THRESHOLD:
        return "small"
    if m >= QUALITY_THRESHOLD:
        return "medium"
    return "large"

final = []
for r in judged_rows:
    label = make_label(r["small_score"], r["medium_score"], r["large_score"])
    final.append({
        "id": r["id"],
        "prompt": r["prompt"],
        "category": r["category"],
        "small_response": r["small_response"],
        "medium_response": r["medium_response"],
        "large_response": r["large_response"],
        "small_score": r["small_score"],
        "medium_score": r["medium_score"],
        "large_score": r["large_score"],
        "routing_label": label,
        "routing_label_id": label2id[label],
    })

df = pd.DataFrame(final)
print(df["routing_label"].value_counts())
print("\nby category:")
print(df.groupby(["category", "routing_label"]).size().unstack(fill_value=0))
df.head(3)


## 5. Push the dataset to `srmty/routing_dataset`

Stratified 80/20 split (fixed seed). The same split is reused for training so
validation numbers line up with what's on the Hub.

In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split as sk_split

ds = Dataset.from_list(final)

idx = list(range(len(ds)))
strat = final and len(set(r["routing_label_id"] for r in final)) > 1
train_idx, val_idx = sk_split(
    idx, test_size=0.2, random_state=SEED,
    stratify=[r["routing_label_id"] for r in final] if strat else None,
)

dsd = DatasetDict({"train": ds.select(sorted(train_idx)),
                   "validation": ds.select(sorted(val_idx))})
print(dsd)

dsd.push_to_hub(DATASET_REPO)
print("pushed ->", f"https://huggingface.co/datasets/{DATASET_REPO}")


## 6. Fine-tune `microsoft/deberta-v3-small` (3-class)

Input is the **prompt only**. At inference the production router has nothing but
the user's prompt - no answers, no scores.

`prompt  ->  DeBERTa-v3-small  ->  [P(small), P(medium), P(large)]`

In [ ]:
import numpy as np
import torch
from torch import nn
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          EarlyStoppingCallback)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=3, id2label=id2label, label2id=label2id,
    torch_dtype=torch.float32,
)

def tok(batch):
    return tokenizer(batch["prompt"], truncation=True, max_length=192)

KEEP = {"input_ids", "attention_mask", "token_type_ids", "labels"}

def prep(split):
    d = split.rename_column("routing_label_id", "labels").map(tok, batched=True)
    return d.remove_columns([c for c in d.column_names if c not in KEEP])

train_ds = prep(dsd["train"])
val_ds   = prep(dsd["validation"])

counts = Counter(dsd["train"]["routing_label_id"])
freq = np.array([counts.get(i, 1) for i in range(3)], dtype=float)
w = np.sqrt(freq.sum() / freq)
w = np.clip(w / w.mean(), 0.5, 3.0)
class_weights = torch.tensor(w, dtype=torch.float32)
print("train label counts:", {i: int(counts.get(i, 0)) for i in range(3)},
      " weights:", [round(x, 2) for x in class_weights.tolist()])


In [ ]:
import math, transformers
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
print("transformers", transformers.__version__)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f1}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.functional.cross_entropy(
            outputs.logits, labels,
            weight=class_weights.to(outputs.logits.device),
            label_smoothing=0.1,
        )
        return (loss, outputs) if return_outputs else loss

EPOCHS, BATCH = 8, 8
total_steps = math.ceil(len(train_ds) / BATCH) * EPOCHS

optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

args = TrainingArguments(
    output_dir="router_out",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=10,
    seed=SEED,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()
print(trainer.evaluate())


## 7. Evaluate the router

Accuracy / precision / recall / F1, confusion matrix, and softmax confidence.
`ROUTER_THRESHOLD` (0.75) decides when a prediction is trusted vs escalated to
the next size up.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

pred = trainer.predict(val_ds)
logits = pred.predictions
y_true = pred.label_ids
y_pred = np.argmax(logits, axis=-1)

print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(LABELS)
ax.set_yticks(range(3)); ax.set_yticklabels(LABELS)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("router confusion matrix")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.show()


In [ ]:
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
conf = probs.max(axis=1)

accepted = conf >= ROUTER_THRESHOLD
print(f"mean confidence           : {conf.mean():.3f}")
print(f"accepted @ {ROUTER_THRESHOLD}          : {accepted.mean()*100:.1f}% of val")
if accepted.any():
    print(f"accuracy on accepted      : {accuracy_score(y_true[accepted], y_pred[accepted]):.3f}")
if (~accepted).any():
    print(f"accuracy on low-confidence: {accuracy_score(y_true[~accepted], y_pred[~accepted]):.3f}")

i = int(np.argmax(conf))
print("\nexample:")
for k, lab in enumerate(LABELS):
    print(f"  {lab:7s} {probs[i][k]:.2f}")
print(f"  -> {LABELS[y_pred[i]]}  (confidence {conf[i]:.2f})")


In [ ]:
NEXT_SIZE = {"small": "medium", "medium": "large", "large": "large"}

@torch.no_grad()
def predict_route(prompt):
    enc = tokenizer(prompt, truncation=True, max_length=256, return_tensors="pt").to(model.device)
    p = torch.softmax(model(**enc).logits[0], dim=-1).tolist()
    pid = int(np.argmax(p))
    predicted, c = LABELS[pid], p[pid]
    chosen = predicted if c >= ROUTER_THRESHOLD else NEXT_SIZE[predicted]
    return {"predicted": predicted, "confidence": round(c, 3), "route_to": chosen,
            "probs": {LABELS[i]: round(p[i], 3) for i in range(3)}}

for q in ["What is 25 * 17?",
          "Prove that the square root of 2 is irrational.",
          "Summarize the plot of Hamlet in two sentences.",
          "Write a Python function that returns the n-th Fibonacci number."]:
    print(q)
    print("   ", predict_route(q), "\n")


## 8. Push the router to `srmty/meridian`

Save model + tokenizer, then push both. The demo app loads **only** this repo;
it calls the Qwen models remotely and never downloads them.

In [ ]:
SAVE_DIR = "meridian-router"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

model.push_to_hub(ROUTER_REPO)
tokenizer.push_to_hub(ROUTER_REPO)
print("pushed ->", f"https://huggingface.co/{ROUTER_REPO}")


## Done

| repo | what |
|---|---|
| [`srmty/routing_dataset`](https://huggingface.co/datasets/srmty/routing_dataset) | prompts + 3x answers + 3x judge scores + `routing_label` |
| [`srmty/meridian`](https://huggingface.co/srmty/meridian) | fine-tuned DeBERTa-v3-small, `prompt -> {small, medium, large}` |

```
meridian_training.ipynb
    +--> srmty/routing_dataset
    +--> srmty/meridian

app  --loads-->  srmty/meridian
     --calls -->  Qwen3 0.6B / 1.7B / 8B  via  router.huggingface.co/v1
```